# 14 · Turning a board into a service

Notebook 13 built a catalogue and evaluated it by typing a command. That is
fine for a laptop and useless at 3am.

A service is three things the command is not: **it answers questions over HTTP,
it runs on a clock, and it keeps what it saw.** This notebook builds all three
and runs them.

![](img/oncall-1-two-services.png)

In [ ]:
import sys; sys.path.insert(0, '..')
from nb import show, sql, fetch, run          # the same helpers as notebooks 1 to 12

---

## Part 1 · Keeping what it saw

A baseline computed from `gold_daily` only works for KPIs that have history in
the warehouse. `records_held` does not: there is no table of what quarantine
used to be.

So the service keeps its own readings, in its own schema.

In [ ]:
from signal_service import store

store.setup()

import psycopg
from pipelines.lib.config import dsn

with psycopg.connect(dsn()) as c:
    rows = c.execute("""
        SELECT table_name FROM information_schema.tables
        WHERE table_schema = 'oncall' ORDER BY table_name
    """).fetchall()
print('the oncall schema:')
for (t,) in rows:
    print('   ', t)

### Read the schema name

The service **reads** `teach` and **writes** `oncall`. It never writes a row
into the warehouse.

That is not tidiness. It is the property that lets you drop the entire
observability layer and rebuild it without touching a single number anybody
reports on. If the signal board could write to `teach`, then every incident
review would have to start with "did the monitoring do this?"

---

## Part 2 · The API

An HTTP surface over the catalogue, so anything can ask without importing
python or knowing which database the numbers live in.

In [ ]:
from signal_service.api import app

for route in app.routes:
    if hasattr(route, 'methods') and route.path != '/openapi.json':
        methods = ','.join(sorted(route.methods - {'HEAD', 'OPTIONS'}))
        print(f'   {methods:5} {route.path}')

### One design decision worth arguing about in class

Look at which of those is a `POST`.

**`GET /signals` measures and tells you. It records nothing and wakes nobody.**
**`POST /evaluate` is the one that can start an investigation.**

A dashboard refreshing every ten seconds must not be able to page somebody. If
measuring had side effects, the act of looking at the board would generate
incidents, and you would have built a machine that alerts on being observed.

## Start it and ask it something

The service is already running in Docker. If it is not, this cell tells you.

In [ ]:
import httpx, os

SIGNAL = os.environ.get('SIGNAL_SERVICE_URL', 'http://localhost:8091')

try:
    health = httpx.get(f'{SIGNAL}/health', timeout=10).json()
    print('health:', health)
except Exception as e:
    print(f'not running: {type(e).__name__}')
    print('start it with:  docker compose -f platform/docker-compose.yml up -d signal-api')
    print('or locally:     uvicorn signal_service.api:app --port 8091')

In [ ]:
kpis = httpx.get(f'{SIGNAL}/kpis', timeout=30).json()
print(f'{len(kpis)} kpis\n')
print(f'  {"name":26} {"owner":15} {"judged by":10} severity')
for k in kpis:
    print(f'  {k["name"]:26} {k["owner"]:15} {k["judgement"]:10} {k["severity"]}')

## The query is not a secret

In [ ]:
d = httpx.get(f'{SIGNAL}/kpis/surge_coverage_pct', timeout=30).json()
print(d['means'], '\n')
print('SQL:', d['sql'])

Anybody arguing with a number deserves to see exactly how it was produced.
Hiding the query is how a metric becomes folklore.

## Evaluate everything, over HTTP

In [ ]:
data = httpx.get(f'{SIGNAL}/signals', timeout=120).json()
print(f'{data["total"]} evaluated, {data["breached"]} breached, '
      f'{data["unmeasured"]} not measurable\n')
for s in data['signals']:
    flag = 'BREACH' if s['breached'] else '  ok  '
    value = '--' if s['value'] is None else f'{s["value"]:,.2f}'
    print(f'  {flag}  {s["kpi"]:26} {value:>13}{s["unit"]:<7} {s["owner"]}')

---

## Part 3 · The clock

This is the whole difference between a board you run and a board that runs.

In [ ]:
import inspect
from signal_service import scheduler

print(inspect.getsource(scheduler.cycle))

### Three decisions in thirty lines

**Why a loop and not cron.** Cron is fine, and for many teams it is the right
answer. A loop is used here because it holds one thing cron cannot: the memory
of what it has already raised. Suppression, backoff and "this is the fourth
time tonight" all need state that survives between cycles, and a process that
exits every minute has none.

**Why it never dies.** Every cycle is wrapped. If the warehouse is down it says
so, waits, and tries again. A scheduler that stops on the first error is a
scheduler that was running yesterday.

**Why it is not the API.** The API answers questions; the clock asks them.
Keeping them apart means a dashboard hammering `/signals` cannot slow the clock,
and a slow clock cannot make the dashboard time out.

## Run one cycle

In [ ]:
run('-m', 'signal_service.scheduler', '--once', '--no-notify')

---

## Part 4 · Not raising the same thing 288 times

The board runs every minute. A broken pipeline stays broken for hours.

Without something, one incident becomes several hundred investigations, and the
thing you built to help becomes the outage.

In [ ]:
from signal_service.kpis import get
from signal_service import evaluate as ev

kpi = get('records_held')
reading, verdict = ev.evaluate(kpi)
breach = ev.to_breach(kpi, reading, verdict)

print('breach id   ', breach.breach_id, '  <- unique every time')
print('fingerprint ', breach.fingerprint(), '  <- the same for the same problem')
print()
again = ev.to_breach(kpi, *ev.evaluate(kpi))
print('a second evaluation of the same problem:')
print('   different id:          ', again.breach_id != breach.breach_id)
print('   identical fingerprint: ', again.fingerprint() == breach.fingerprint())

The fingerprint is the KPI, the direction and the value. **The same KPI breaching
the same way is the same incident**, and the store refuses to open a second one
within a window.

---

## Part 5 · The record, and the doorbell

Two things happen when a signal breaches, and they are not the same.

In [ ]:
print(inspect.getsource(__import__('signal_service.emit', fromlist=['emit']).emit))

### Read the order

**RECORD** the breach to `oncall.breaches`. Durable. This is the truth.
**NOTIFY** the agent service over HTTP. A doorbell. This is a courtesy.

Do them in that order and a failed notification costs **latency**, not an
incident: the record is on disk and the agent picks it up on its next sweep. Do
it the other way round and a restart at the wrong second means a breach nobody
ever hears about, with no error anywhere.

That is the same argument as writing rows before committing a Kafka offset, and
it turns up every time two systems have to agree on something.

---

## What you learned

- A service is **an API, a clock, and a memory**. The command was none of those
- The board **reads the warehouse and writes its own schema**, never the reverse
- **`GET` must not have side effects.** A dashboard cannot be allowed to page people
- The clock exists so nobody has to type, and it **never dies on an error**
- **Deduplicate by fingerprint**, or one broken pipeline becomes 288 incidents
- **Write the record, then ring the doorbell.** Never the other way round